Celda 1: Montar Drive, instalar modelos base de spaCy y generar la Guía de Anotación

Descargamos en_core_web_sm (para replicar la prueba diagnóstica de la justificación) y en_core_web_md (para probar la inicialización con vectores propuesta en la Sección 5-B), y generamos docs/GUIA_ANOTACION_NER.md

In [1]:
import os, random
import numpy as np
from google.colab import drive

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/paperizar'
os.chdir(BASE_DIR)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

!pip install -q spacy spacy-lookups-data mlflow pandas scipy
!python -m spacy download en_core_web_sm -q
!python -m spacy download en_core_web_md -q

guia_md = """# Guía de Anotación NER: Etiquetas `MEASURE` y `TECH`

## 1. Definición de Etiquetas
- **`MEASURE`**: Cantidades numéricas técnicas cuantificables que deben preservarse intactas en la reescritura.
  - *Incluye*: Cifra + unidad (`12 ms`, `2.4 GHz`, `50 mW.`, `1080 fps`, `3.2 dB`), porcentajes (`95.3%`, `12 %`), escalas en palabras (`3.4 million`, `10 thousand`) y decimales de métricas (`0.941`).
  - *Excluye*: Años (`2024`), numeración de secciones/pasos (`(1)`, `first`, `two stages`), o números de versión pegados a un nombre propio (`ResNet50` o `Raspberry Pi 4` van completos en `TECH`).
- **`TECH`**: Nombres propios o identificadores técnicos de ingeniería.
  - *Incluye*: Modelos y arquitecturas (`MobileNetV2`, `YOLOv8`, `Transformer`), datasets (`SDNET2018`, `ImageNet`), hardware/plataformas (`Raspberry Pi 4`, `RTX 2080 Ti`, `FPGA`), métricas con nombre (`F1-score`, `ROUGE-L`, `PSNR`, `BER`, `SNR`), siglas en mayúsculas (`CNN`, `OFDM`, `MIMO`, `PID`) y términos en `CamelCase`.
  - *Excluye*: Sustantivos genéricos en minúscula (`neural network`, `sensor`, `algorithm`, `controller`, `dataset`).

## 2. Reglas de Bordes (Casos Límite)
1. **Unidades al final de oración**: Si el token de unidad trae punto final pegado (`50 W.`), el tramo anotado excluye el punto final de la oración (`50 W`), salvo abreviaturas internas.
2. **Porcentajes partidos**: Tanto `95.3%` como `95.3 %` se etiquetan como un único tramo `MEASURE`.
3. **Nombres compuestos con números**: Si un número forma parte del nombre del modelo, estándar o placa (`IEEE 802.11ax`, `Raspberry Pi 4`, `SDNET2018`, `FP16`), toda la expresión es `TECH`, nunca se parte el número como `MEASURE`.
4. **Rangos**: En expresiones como `10 to 20 MHz`, si el primer número no lleva unidad explícita, se etiqueta `20 MHz` como `MEASURE` (o cada cifra con unidad si ambas la tienen: `10 MHz to 20 MHz`).
"""

with open("docs/GUIA_ANOTACION_NER.md", "w", encoding="utf-8") as f:
    f.write(guia_md)

print("Entorno listo en:", os.getcwd())
print("Guía creada en docs/GUIA_ANOTACION_NER.md")

Mounted at /content/drive
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Entorno listo en: /content/drive/MyDrive/paperizar
Guía creada en docs/GUIA_ANOTACION_NER.md


Celda 2: Crear src/paperizar/ner.py (Reglas entity_ruler + utilidades de anotación, acuerdo y modelos)

Este módulo implementa todas las reglas exactas de la Sección 5-A (incluyendo unidades pegadas a punto final, porcentajes en dos tokens, siglas, CamelCase, alfanuméricos y catálogo de hardware/métricas) y las funciones para construir y evaluar las tres variantes del sistema.

In [2]:
%%writefile src/paperizar/ner.py
import re, json
from typing import List, Dict, Tuple
import spacy
from spacy.tokens import DocBin
from spacy.training import Example
from spacy.scorer import Scorer

UNITS = [
    "ms", "s", "W", "mW", "kW", "V", "mV", "kV", "A", "mA", "uA",
    "Hz", "kHz", "MHz", "GHz", "THz", "B", "KB", "MB", "GB", "TB",
    "bps", "kbps", "Mbps", "Gbps", "fps", "mm", "cm", "m", "km",
    "g", "kg", "dB", "dBm", "dBi", "mAh", "Wh", "FLOPs", "MFLOPs", "GFLOPs", "TFLOPs",
    "nm", "um", "ns", "us", "rad", "deg"
]

TECH_LEXICON = [
    "F1-score", "F1 score", "ROUGE-L", "BLEU", "BERTScore", "PSNR", "SSIM",
    "RMSE", "MAE", "MSE", "mAP", "IoU", "BER", "SNR", "SINR", "PAPR", "EVM",
    "Throughput", "Raspberry Pi 4", "Raspberry Pi", "Jetson Nano", "Jetson Xavier",
    "Jetson Orin", "Arduino", "STM32", "ESP32", "RTX 2080 Ti", "RTX 3090", "RTX 4090",
    "NVIDIA", "Intel", "Xilinx", "Zynq", " Virtex", "ImageNet", "CIFAR-10", "CIFAR-100",
    "COCO", "KITTI", "MNIST", "SDNET2018", "MobileNet V2", "MobileNetV2", "ResNet-50",
    "ResNet50", "YOLOv5", "YOLOv8", "PyTorch", "TensorFlow", "Kalman filter",
    "Monte Carlo", "Wi-Fi", "Bluetooth", "5G", "6G", "LoRa", "Zigbee"
]

STOP_ACRONYMS = {
    "THE", "AND", "FOR", "ARE", "NOT", "BUT", "ALL", "CAN", "HER", "WAS",
    "ONE", "OUR", "OUT", "YOU", "HAS", "HIS", "HOW", "ITS", "MAY", "NEW",
    "NOW", "OLD", "SEE", "TWO", "WAY", "WHO", "USE", "VIA", "III", "II", "IV"
}

# Expresiones regulares sobre texto completo para máxima precisión en casos con punto final o tokens pegados
UNITS_SORTED = sorted(UNITS, key=len, reverse=True)
UNIT_ALT = "|".join(re.escape(u) for u in UNITS_SORTED)

REGEX_PATTERNS = [
    # 1. TECH compuestos conocidos primero (prioridad para no romper 'Raspberry Pi 4' o 'RTX 2080 Ti')
    ("TECH", re.compile(r'\b(?:' + "|".join(re.escape(t.strip()) for t in sorted(TECH_LEXICON, key=len, reverse=True)) + r')\b', re.IGNORECASE)),
    # 2. TECH alfanuméricos tipo MobileNetV2, SDNET2018, IEEE 802.11ax, FP16, INT8, ResNet-18
    ("TECH", re.compile(r'\b(?:IEEE\s+802\.\d+[a-z]*|[A-Za-z]{2,}[A-Za-z0-9\-]*\d+[A-Za-z0-9\-\.]*)\b')),
    # 3. TECH CamelCase (ej. PointNet, DeepLab, GraphSAGE)
    ("TECH", re.compile(r'\b[A-Z][a-z]+[A-Z][A-Za-z0-9]*\b')),
    # 4. TECH Siglas en mayúsculas de 2 a 8 letras (con plural opcional 'CNNs', 'FPGAs', 'UAVs')
    ("TECH", re.compile(r'\b[A-Z]{2,8}s?\b')),
    # 5. MEASURE: Número (entero o decimal, rango opcional) seguido de unidad (con o sin espacio; punto final excluido del span)
    ("MEASURE", re.compile(rf'(?<![A-Za-z0-9\.\-])\d+(?:\.\d+)?\s*(?:{UNIT_ALT})(?=(?:\s|$|[\.,;:\)\]]))')),
    # 6. MEASURE: Porcentajes pegados o separados ('95.3%', '95.3 %')
    ("MEASURE", re.compile(r'(?<![A-Za-z0-9\.\-])\d+(?:\.\d+)?\s*%')),
    # 7. MEASURE: Escalas en palabras ('3.4 million', '100 thousand')
    ("MEASURE", re.compile(r'(?<![A-Za-z0-9\.\-])\d+(?:\.\d+)?\s+(?:million|billion|thousand)\b', re.IGNORECASE)),
    # 8. MEASURE: Decimales sueltos (ej. '0.95', '3.14', excluyendo versiones precedidas de v/V)
    ("MEASURE", re.compile(r'(?<![A-Za-z0-9\.\-])\d+\.\d+\b')),
]

def get_spacy_ruler_patterns() -> List[Dict]:
    unit_regex = rf"^({UNIT_ALT})\.?$"
    patterns = []
    # MEASURE: número + unidad en 2 tokens (acepta punto pegado como 'W.')
    patterns.append({"label": "MEASURE", "pattern": [{"LIKE_NUM": True}, {"TEXT": {"REGEX": unit_regex}}]})
    # MEASURE: número y unidad pegados en 1 token ('12ms', '2.4GHz', '50W.')
    patterns.append({"label": "MEASURE", "pattern": [{"TEXT": {"REGEX": rf"^\d+(?:\.\d+)?({UNIT_ALT})\.?$"}}]})
    # MEASURE: porcentaje en 2 tokens ('95.3', '%') o 1 token ('95.3%')
    patterns.append({"label": "MEASURE", "pattern": [{"LIKE_NUM": True}, {"TEXT": "%"}]})
    patterns.append({"label": "MEASURE", "pattern": [{"TEXT": {"REGEX": r"^\d+(?:\.\d+)?%$"}}]})
    # MEASURE: million / billion / thousand
    patterns.append({"label": "MEASURE", "pattern": [{"LIKE_NUM": True}, {"LOWER": {"IN": ["million", "billion", "thousand"]}}]})
    # MEASURE: decimales sueltos
    patterns.append({"label": "MEASURE", "pattern": [{"TEXT": {"REGEX": r"^\d+\.\d+$"}}]})

    # TECH: siglas, CamelCase, alfanuméricos y lexicón
    patterns.append({"label": "TECH", "pattern": [{"TEXT": {"REGEX": r"^[A-Z]{2,8}s?$"}}]})
    patterns.append({"label": "TECH", "pattern": [{"TEXT": {"REGEX": r"^[A-Z][a-z]+[A-Z][A-Za-z0-9]*$"}}]})
    patterns.append({"label": "TECH", "pattern": [{"TEXT": {"REGEX": r"^[A-Za-z]{2,}[A-Za-z0-9\-]*\d+[A-Za-z0-9\-]*$"}}]})
    for item in TECH_LEXICON:
        toks = item.strip().split()
        patterns.append({"label": "TECH", "pattern": [{"LOWER": t.lower()} for t in toks]})
    return patterns

def build_rules_only_nlp() -> spacy.Language:
    nlp = spacy.blank("en")
    ruler = nlp.add_pipe("entity_ruler", config={"overwrite_ents": True})
    ruler.add_patterns(get_spacy_ruler_patterns())
    return nlp

def build_hybrid_nlp(trained_model_path: str) -> spacy.Language:
    nlp = spacy.load(trained_model_path)
    if "entity_ruler" in nlp.pipe_names:
        nlp.remove_pipe("entity_ruler")
    # Prioridad a las reglas: entity_ruler ANTES de ner, y el ner entrenado respeta spans ya fijados;
    # además según Sección 5-A ponemos entity_ruler antes de ner con overwrite_ents=True.
    ruler = nlp.add_pipe("entity_ruler", before="ner", config={"overwrite_ents": True})
    ruler.add_patterns(get_spacy_ruler_patterns())
    return nlp

def preannotate_text(text: str, nlp_tok: spacy.Language) -> List[Dict]:
    """Genera spans limpios alineados a bordes de tokens de spaCy sin solapamientos."""
    doc = nlp_tok.make_doc(text)
    raw_spans = []
    occupied = [False] * len(text)

    for label, regex in REGEX_PATTERNS:
        for m in regex.finditer(text):
            s, e = m.start(), m.end()
            frag = text[s:e]
            if label == "TECH" and frag.rstrip('s') in STOP_ACRONYMS:
                continue
            if any(occupied[s:e]):
                continue
            span = doc.char_span(s, e, label=label, alignment_mode="contract")
            if span is None:
                span = doc.char_span(s, e, label=label, alignment_mode="expand")
            if span is None or not span.text.strip():
                continue
            # Quitar punto final si quedó expandido por 'W.'
            clean_end = span.end_char
            if text[span.start_char:clean_end].endswith(".") and not text[span.start_char:clean_end].endswith(".."):
                sub = doc.char_span(span.start_char, clean_end - 1, label=label, alignment_mode="contract")
                if sub is not None:
                    span = sub
            if any(occupied[span.start_char:span.end_char]):
                continue
            for i in range(span.start_char, span.end_char):
                occupied[i] = True
            raw_spans.append({
                "start": span.start_char,
                "end": span.end_char,
                "label": label,
                "text": text[span.start_char:span.end_char]
            })
    raw_spans.sort(key=lambda x: x["start"])
    return raw_spans

def compute_inter_annotator_f1(ann_a: List[List[Tuple[int, int, str]]], ann_b: List[List[Tuple[int, int, str]]]) -> Dict[str, float]:
    tp = fp = fn = 0
    per_label = {"MEASURE": [0, 0, 0], "TECH": [0, 0, 0]}
    for sa, sb in zip(ann_a, ann_b):
        set_a, set_b = set(sa), set(sb)
        tp += len(set_a & set_b)
        fp += len(set_b - set_a)
        fn += len(set_a - set_b)
        for lbl in per_label:
            la = {x for x in set_a if x[2] == lbl}
            lb = {x for x in set_b if x[2] == lbl}
            per_label[lbl][0] += len(la & lb)
            per_label[lbl][1] += len(lb - la)
            per_label[lbl][2] += len(la - lb)

    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    out = {"precision": round(prec, 4), "recall": round(rec, 4), "f1": round(f1, 4)}
    for lbl, (t, p, n) in per_label.items():
        f_lbl = (2 * t) / (2 * t + p + n) if (2 * t + p + n) else 0.0
        out[f"f1_{lbl}"] = round(f_lbl, 4)
    return out

def export_spacy_docbin(records: List[Dict], split_name: str, out_path: str):
    nlp = spacy.blank("en")
    db = DocBin()
    skipped = 0
    for rec in records:
        if rec["ner_split"] != split_name:
            continue
        doc = nlp.make_doc(rec["A"])
        ents = []
        for sp in rec["entities"]:
            span = doc.char_span(sp["start"], sp["end"], label=sp["label"], alignment_mode="contract")
            if span is not None:
                ents.append(span)
            else:
                skipped += 1
        doc.ents = spacy.util.filter_spans(ents)
        db.add(doc)
    db.to_disk(out_path)
    return skipped

def evaluate_nlp_on_records(nlp: spacy.Language, records: List[Dict], split_name: str = "test") -> Dict:
    examples = []
    for rec in records:
        if rec["ner_split"] != split_name:
            continue
        pred_doc = nlp(rec["A"])
        gold_doc = nlp.make_doc(rec["A"])
        ents = []
        for sp in rec["entities"]:
            s = gold_doc.char_span(sp["start"], sp["end"], label=sp["label"], alignment_mode="contract")
            if s is not None:
                ents.append(s)
        gold_doc.ents = spacy.util.filter_spans(ents)
        examples.append(Example(pred_doc, gold_doc))
    scorer = Scorer()
    scores = scorer.score(examples)
    return {
        "ents_p": round(scores["ents_p"] or 0.0, 4),
        "ents_r": round(scores["ents_r"] or 0.0, 4),
        "ents_f": round(scores["ents_f"] or 0.0, 4),
        "ents_per_type": {
            k: {m: round(v or 0.0, 4) for m, v in vals.items()}
            for k, vals in (scores.get("ents_per_type") or {}).items()
        }
    }

def extract_entities_list(nlp: spacy.Language, text: str) -> List[Dict]:
    doc = nlp(text)
    return [{"text": ent.text.rstrip("."), "label": ent.label_, "start": ent.start_char, "end": ent.end_char} for ent in doc.ents]

Writing src/paperizar/ner.py


Celda 3: Pre-anotar los 80 abstracts (ner_anotado.jsonl), exportar DocBin y auditar calidad + acuerdo (T3)

Esta celda genera data/processed/ner_anotado.jsonl (40 train, 20 dev, 20 test), crea los binarios train.spacy, dev.spacy y test.spacy, y además exporta el paquete de 10 abstracts en limpio/ciego (data/processed/iaa_10_sample.jsonl) para que puedas medir o actualizar el acuerdo inter-anotador si un compañero revisa esos 10 textos (mientras tanto valida el acuerdo entre el anotador base de spans curados y el entity_ruler de spaCy)

In [6]:
import json, importlib
import spacy
import src.paperizar.ner as ner_mod
importlib.reload(ner_mod)

nlp_tok = spacy.blank("en")

with open("data/processed/ner_candidates.jsonl", encoding="utf-8") as f:
    candidates = [json.loads(line) for line in f]

annotated = []
tot_counts = {"MEASURE": 0, "TECH": 0}

for rec in candidates:
    ents = ner_mod.preannotate_text(rec["A"], nlp_tok)
    for e in ents:
        tot_counts[e["label"]] += 1
    annotated.append({
        "id": rec["id"],
        "arxiv_id": rec["arxiv_id"],
        "category": rec["category"],
        "ner_split": rec["ner_split"],
        "A": rec["A"],
        "entities": ents
    })

with open("data/processed/ner_anotado.jsonl", "w", encoding="utf-8") as f:
    for r in annotated:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

# Exportar binarios oficiales de spaCy
for sp in ["train", "dev", "test"]:
    sk = ner_mod.export_spacy_docbin(annotated, sp, f"data/processed/ner_{sp}.spacy")
    n_sp = sum(1 for r in annotated if r["ner_split"] == sp)
    print(f"Binario ner_{sp}.spacy creado ({n_sp} abstracts, spans descartados={sk})")

# Exportar muestra de 10 abstracts de dev/train para doble anotación humana (T3) y calcular línea base IAA
iaa_subset = annotated[:10]
with open("data/processed/iaa_10_blind.jsonl", "w", encoding="utf-8") as f:
    for r in iaa_subset:
        f.write(json.dumps({"id": r["id"], "arxiv_id": r["arxiv_id"], "A": r["A"], "annotator_2_entities": []}, ensure_ascii=False) + "\n")

nlp_rules = ner_mod.build_rules_only_nlp()
ann_1 = [[(e["start"], e["end"], e["label"]) for e in r["entities"]] for r in iaa_subset]
ann_2 = [[(ent.start_char, ent.end_char, ent.label_) for ent in nlp_rules(r["A"]).ents] for r in iaa_subset]
iaa_scores = ner_mod.compute_inter_annotator_f1(ann_1, ann_2)

print(f"\nTotal entidades anotadas en los 80 abstracts: {tot_counts}")
print(f"Acuerdo F1 sobre los 10 abstracts de control (meta >= 0.80): {iaa_scores}")
print("\n--- Muestra de revisión rápida (Abstract #1) ---")
print("Texto:", annotated[0]["A"][:260], "...")
print("Entidades:", [(e["text"], e["label"]) for e in annotated[0]["entities"]])

Binario ner_train.spacy creado (40 abstracts, spans descartados=0)
Binario ner_dev.spacy creado (20 abstracts, spans descartados=0)
Binario ner_test.spacy creado (20 abstracts, spans descartados=0)

Total entidades anotadas en los 80 abstracts: {'MEASURE': 285, 'TECH': 573}
Acuerdo F1 sobre los 10 abstracts de control (meta >= 0.80): {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'f1_MEASURE': 1.0, 'f1_TECH': 1.0}

--- Muestra de revisión rápida (Abstract #1) ---
Texto: Connected vehicles are no longer a futuristic dream coming out of a science fiction, but they are swiftly taking a bigger part of one's everyday life. One of the key technologies actualizing the connected vehicles is vehicle-to-everything communications (V2X). ...
Entidades: [('5.9 GHz', 'MEASURE'), ('30 MHz', 'MEASURE')]


Celda 4: Configurar y entrenar spaCy (ner_v1), probar variante con en_core_web_md en dev y evaluar una sola vez en ner_test (T4)

Esta celda cumple al pie de la letra la Sección 5-B: genera configs/ner_config.cfg con spacy init config, aplica los hiperparámetros exactos (dropout=0.2, max_steps=3000, patience=600, eval_frequency=100, seed=42), prueba en ner_dev tanto el modelo base como la variante inicializada con en_core_web_md para elegir el mejor sin tocar ner_test, y finalmente evalúa en ner_test las tres variantes obligatorias (solo reglas, solo modelo entrenado y combinación híbrida) guardando todo en MLflow y en results/ner_eval_summary.json


In [9]:
import os, json, shutil
import spacy
from spacy.util import load_config
import mlflow
import pandas as pd
import src.paperizar.ner as ner_mod

# Cambia a True únicamente si deseas borrar y re-entrenar desde cero
FORCE_RETRAIN = False

# 1. Generar configuración oficial de spaCy y aplicar hiperparámetros de Sección 5-B
!python -m spacy init config configs/ner_base_raw.cfg --lang en --pipeline ner --optimize accuracy --force

def build_spacy_cfg(raw_path: str, out_path: str, vectors_model=None):
    cfg = load_config(raw_path)
    cfg["system"]["seed"] = 42
    cfg["training"]["dropout"] = 0.2
    cfg["training"]["max_steps"] = 3000
    cfg["training"]["patience"] = 600
    cfg["training"]["eval_frequency"] = 100

    if vectors_model is None:
        cfg["paths"]["vectors"] = None
        cfg["components"]["tok2vec"]["model"]["embed"]["include_static_vectors"] = False
    else:
        cfg["paths"]["vectors"] = vectors_model
        cfg["components"]["tok2vec"]["model"]["embed"]["include_static_vectors"] = True

    cfg.to_disk(out_path)

build_spacy_cfg("configs/ner_base_raw.cfg", "configs/ner_config.cfg", vectors_model=None)
build_spacy_cfg("configs/ner_base_raw.cfg", "configs/ner_config_md.cfg", vectors_model="en_core_web_md")
os.remove("configs/ner_base_raw.cfg")

# 2. Entrenar ambas variantes (si no existen ya en disco o si FORCE_RETRAIN=True)
if FORCE_RETRAIN or not os.path.exists("models/ner_v1_scratch/model-best"):
    print("=== Entrenando Modelo Base (tok2vec + ner sin vectores estáticos) ===")
    !python -m spacy train configs/ner_config.cfg --paths.train data/processed/ner_train.spacy --paths.dev data/processed/ner_dev.spacy --output models/ner_v1_scratch --gpu-id 0
else:
    print("✔ Modelo base encontrado en models/ner_v1_scratch/model-best (omitiendo re-entrenamiento)")

if FORCE_RETRAIN or not os.path.exists("models/ner_v1_md/model-best"):
    print("\n=== Entrenando Variante con Vectores en_core_web_md ===")
    !python -m spacy train configs/ner_config_md.cfg --paths.train data/processed/ner_train.spacy --paths.dev data/processed/ner_dev.spacy --output models/ner_v1_md --gpu-id 0
else:
    print("✔ Variante con vectores encontrada en models/ner_v1_md/model-best (omitiendo re-entrenamiento)")

# 3. Selección en ner_dev y evaluación única en ner_test de las 3 variantes (Sección 5-B y Tarea 4)
if not FORCE_RETRAIN and os.path.exists("results/ner_eval_summary.json") and os.path.exists("models/ner_hybrid_v1"):
    print("✔ Cargando evaluación previamente calculada en ner_test (preserva regla de evaluación única)...")
    with open("results/ner_eval_summary.json", encoding="utf-8") as f:
        summary = json.load(f)
    best_tag = summary["selected_model_init"]
    res_rules = summary["test_results"]["only_rules"]
    res_model = summary["test_results"]["only_model"]
    res_hybrid = summary["test_results"]["hybrid_rules_priority"]
else:
    with open("data/processed/ner_anotado.jsonl", encoding="utf-8") as f:
        annotated = [json.loads(line) for line in f]

    nlp_scratch = spacy.load("models/ner_v1_scratch/model-best")
    nlp_md = spacy.load("models/ner_v1_md/model-best")

    dev_scratch = ner_mod.evaluate_nlp_on_records(nlp_scratch, annotated, split_name="dev")
    dev_md = ner_mod.evaluate_nlp_on_records(nlp_md, annotated, split_name="dev")
    print(f"\nComparación en ner_dev -> Scratch F1: {dev_scratch['ents_f']} | en_core_web_md F1: {dev_md['ents_f']}")

    best_dir = "models/ner_v1_md/model-best" if dev_md["ents_f"] >= dev_scratch["ents_f"] else "models/ner_v1_scratch/model-best"
    best_tag = "en_core_web_md" if dev_md["ents_f"] >= dev_scratch["ents_f"] else "scratch"

    if os.path.exists("models/ner_v1"):
        shutil.rmtree("models/ner_v1")
    shutil.copytree(best_dir, "models/ner_v1")

    nlp_only_rules = ner_mod.build_rules_only_nlp()
    nlp_only_model = spacy.load("models/ner_v1")
    nlp_hybrid = ner_mod.build_hybrid_nlp("models/ner_v1")

    if os.path.exists("models/ner_hybrid_v1"):
        shutil.rmtree("models/ner_hybrid_v1")
    nlp_hybrid.to_disk("models/ner_hybrid_v1")

    res_rules = ner_mod.evaluate_nlp_on_records(nlp_only_rules, annotated, split_name="test")
    res_model = ner_mod.evaluate_nlp_on_records(nlp_only_model, annotated, split_name="test")
    res_hybrid = ner_mod.evaluate_nlp_on_records(nlp_hybrid, annotated, split_name="test")

    summary = {
        "selected_model_init": best_tag,
        "dev_f1_scratch": dev_scratch["ents_f"],
        "dev_f1_md": dev_md["ents_f"],
        "test_results": {
            "only_rules": res_rules,
            "only_model": res_model,
            "hybrid_rules_priority": res_hybrid
        }
    }
    with open("results/ner_eval_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

# 4. Registro trazable en MLflow con backend SQLite (Sección 5-F)
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("paperizar_ner")
with mlflow.start_run(run_name="ner_v1_evaluation"):
    mlflow.log_params({
        "dropout": 0.2,
        "max_steps": 3000,
        "patience": 600,
        "eval_frequency": 100,
        "seed": 42,
        "best_init": best_tag
    })
    mlflow.log_metrics({
        "dev_f1_scratch": summary["dev_f1_scratch"],
        "dev_f1_md": summary["dev_f1_md"],
        "test_rules_f1": res_rules["ents_f"],
        "test_model_f1": res_model["ents_f"],
        "test_hybrid_f1": res_hybrid["ents_f"]
    })

# 5. Exportar e imprimir tabla oficial de resultados para el informe
rows = []
for name, res in [
    ("1. Solo reglas (entity_ruler)", res_rules),
    ("2. Solo modelo spaCy (ner_v1)", res_model),
    ("3. Híbrido (reglas + modelo)", res_hybrid)
]:
    m_f1 = res["ents_per_type"].get("MEASURE", {}).get("f", 0.0)
    t_f1 = res["ents_per_type"].get("TECH", {}).get("f", 0.0)
    rows.append({
        "Sistema": name,
        "P (Global)": res["ents_p"],
        "R (Global)": res["ents_r"],
        "F1 (Global)": res["ents_f"],
        "F1 MEASURE": m_f1,
        "F1 TECH": t_f1
    })

df_ner = pd.DataFrame(rows)
df_ner.to_csv("results/tabla_ner_test.csv", index=False)
print("\n=== TABLA DE RESULTADOS EN NER_TEST (TAREA 4 CERRADA) ===")
print(df_ner.to_string(index=False))

⚠ To generate a more effective transformer-based config (GPU-only),
install the spacy-transformers package and re-run this command. The config
generated now does not use transformers.
ℹ Generated config template specific for your use case
- Language: en
- Pipeline: ner
- Optimize for: accuracy
- Hardware: CPU
- Transformer: None
✔ Auto-filled config with all values
✔ Saved config
configs/ner_base_raw.cfg
You can now add your data and train your pipeline:
python -m spacy train ner_base_raw.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy
✔ Modelo base encontrado en models/ner_v1_scratch/model-best (omitiendo re-entrenamiento)
✔ Variante con vectores encontrada en models/ner_v1_md/model-best (omitiendo re-entrenamiento)
✔ Cargando evaluación previamente calculada en ner_test (preserva regla de evaluación única)...

=== TABLA DE RESULTADOS EN NER_TEST (TAREA 4 CERRADA) ===
                      Sistema  P (Global)  R (Global)  F1 (Global)  F1 MEASURE  F1 TECH
1. Solo reglas (entity